# Dataset Exploration: CADCODER/GenCAD-Code

147K paired examples of 2D images and [CadQuery](https://cadquery.readthedocs.io/) Python scripts, from [CADCODER/GenCAD-Code](https://huggingface.co/datasets/CADCODER/GenCAD-Code). This notebook looks at what's actually in the data before any modeling — sample image/code pairs, how long the code scripts are, and which geometric operations dominate — since several choices made elsewhere in this project (`MAX_SEQ_LEN`, image resolution, tokenizer vocab size) are directly justified by what's found here.

No training happens in this notebook. See `01_train_models.ipynb` for that.

In [ ]:
import os, pathlib, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/adebowalep/image-to-cadquery.git"  # <-- set this (private repo: https://<token>@github.com/...)
    repo_dir = pathlib.Path("/content") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)

    # System library CadQuery's VTK dependency may need on headless Linux (the original Colab
    # notebook installed it too). Harmless if already present.
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "libgl1"], check=False)

    # Colab already ships torch/torchvision/transformers/pandas/matplotlib -- install only what it lacks.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "cadquery==2.6.0", "cadquery-ocp==7.8.1.1", "trimesh", "datasets"],
        check=True,
    )

_REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_REPO_ROOT))          # `metrics/`
sys.path.insert(0, str(_REPO_ROOT / "src"))  # `cad_code_gen` (no `pip install -e` needed)

# Everything that must outlive the session (dataset cache, tokenizer, checkpoints, results)
# lives under STORAGE_ROOT: Google Drive on Colab, the repo itself locally.
STORAGE_ROOT = pathlib.Path("/content/drive/MyDrive/cad_code_gen") if IN_COLAB else _REPO_ROOT
STORAGE_ROOT.mkdir(parents=True, exist_ok=True)

# The Hugging Face dataset cache is memory-mapped and read at random while training, which is slow
# and flaky over the Drive mount. On Colab keep it on the VM's local disk (~1 GB, re-downloaded
# each new session); only checkpoints, tokenizer and results need to survive on Drive.
CACHE_DIR = pathlib.Path("/content/hf_cache") if IN_COLAB else STORAGE_ROOT / "huggingface_cache"
print("repo:", _REPO_ROOT, "| storage:", STORAGE_ROOT, "| dataset cache:", CACHE_DIR)

In [ ]:
import re
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset

from cad_code_gen.data import read_pil_image
from cad_code_gen.config import TrainConfig

## Load the dataset & reproduce the train/val/test split

Same `SEED` and 90/10 split `01_train_models.ipynb` uses, so the numbers here describe exactly what the model is trained/evaluated on — not a different sample.

In [ ]:
SEED = 42
ds_train_full, ds_test = load_dataset(
    "CADCODER/GenCAD-Code", num_proc=16, split=["train", "test"], cache_dir=str(CACHE_DIR)
)
split = ds_train_full.shuffle(seed=SEED).train_test_split(test_size=0.10, seed=SEED)
hf_train, hf_val = split["train"], split["test"]

print(f"train : {len(hf_train):>7,}")
print(f"val   : {len(hf_val):>7,}")
print(f"test  : {len(ds_test):>7,}")
print(f"total : {len(hf_train) + len(hf_val) + len(ds_test):>7,}")

In [ ]:
df_train = hf_train.select(range(min(5000, len(hf_train)))).to_pandas()  # a sample is plenty for EDA
df_train.head()

## A single example: image + generating code

In [ ]:
row = df_train.iloc[0]
image = read_pil_image(row["image"])

plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.axis("off")
plt.title(f"deepcad_id: {row['deepcad_id']}")
plt.show()

print(row["cadquery"])

## A grid of examples

In [ ]:
def show_image_grid(df, n=9, figsize_per_cell=3):
    n = min(n, len(df))
    cols = 3
    rows = -(-n // cols)  # ceil
    fig, axes = plt.subplots(rows, cols, figsize=(cols * figsize_per_cell, rows * figsize_per_cell))
    axes = axes.flatten()

    for i in range(n):
        img = read_pil_image(df.iloc[i]["image"])
        axes[i].imshow(img)
        axes[i].axis("off")
        axes[i].set_title(str(df.iloc[i]["deepcad_id"]), fontsize=8)
    for j in range(n, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

show_image_grid(df_train, n=9)

## Code length distribution

Justifies `TrainConfig.max_seq_len` (currently `256`, used everywhere the tokenizer truncates a script): if most scripts are far shorter than that, the budget is generous; if a meaningful fraction get cut off, that's a real source of training-signal loss worth knowing about.

In [ ]:
char_lengths = df_train["cadquery"].str.len()
line_counts = df_train["cadquery"].str.count("\n") + 1

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(char_lengths, bins=40)
axes[0].set_title("Code length (characters)"); axes[0].set_xlabel("characters")
axes[1].hist(line_counts, bins=40)
axes[1].set_title("Code length (lines)"); axes[1].set_xlabel("lines")
plt.tight_layout()
plt.show()

print(char_lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

## Which CadQuery operations dominate?

A rough proxy for "geometric vocabulary" the model has to learn — shapes built mostly from `box`/`circle`/`extrude` are structurally simpler than ones relying heavily on `threePointArc`/`union`/`cut` (boolean ops, curved profiles).

In [ ]:
OPERATIONS = [
    "box", "circle", "rect", "extrude", "hole", "cboreHole", "union", "cut",
    "threePointArc", "lineTo", "moveTo", "workplane", "faces", "revolve", "loft",
]

counts = Counter()
for code_str in df_train["cadquery"]:
    for op in OPERATIONS:
        counts[op] += len(re.findall(rf"\.{op}\(", code_str))

op_series = pd.Series(counts).sort_values(ascending=False)
op_series.plot.bar(figsize=(10, 4), title="Operation frequency across sampled training scripts")
plt.ylabel("occurrences")
plt.tight_layout()
plt.show()

## Takeaways

Fill in once the cells above have actually run against the real dataset:
- Is `max_seq_len=256` generous enough, or does a meaningful tail of scripts get truncated?
- Which operations are rare enough that the model may simply never learn them well from this data alone?
- Anything about the images themselves (resolution, background, lighting) worth adjusting the preprocessing in `cad_code_gen.data.build_image_transform` for?

Next: `01_train_models.ipynb` trains every model architecture on this data.